In [17]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("HousePrice_Null_Handling") \
    .getOrCreate()

# Đọc file CSV
df = spark.read.csv(
    "house_prices.csv", 
    header=True, 
    inferSchema=True, 
    escape="\"",       # <-- RẤT QUAN TRỌNG: để Spark hiểu "" là ngoặc kép trong text
    multiLine=True
)

spark.conf.set("spark.sql.ansi.enabled", "false")

# Chuẩn hóa tên cột
for col_name in df.columns:
    clean_name = col_name.strip().replace(" ", "_").replace("(", "").replace(")", "").lower()
    df = df.withColumnRenamed(col_name, clean_name)

df.printSchema()

root
 |-- index: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- description: string (nullable = true)
 |-- amountin_rupees: string (nullable = true)
 |-- price_in_rupees: integer (nullable = true)
 |-- location: string (nullable = true)
 |-- carpet_area: string (nullable = true)
 |-- status: string (nullable = true)
 |-- floor: string (nullable = true)
 |-- transaction: string (nullable = true)
 |-- furnishing: string (nullable = true)
 |-- facing: string (nullable = true)
 |-- overlooking: string (nullable = true)
 |-- society: string (nullable = true)
 |-- bathroom: string (nullable = true)
 |-- balcony: string (nullable = true)
 |-- car_parking: string (nullable = true)
 |-- ownership: string (nullable = true)
 |-- super_area: string (nullable = true)
 |-- dimensions: string (nullable = true)
 |-- plot_area: string (nullable = true)



In [8]:
def check_null_summary(dataframe):
    total_rows = dataframe.count()
    
    # Check null, NaN
    null_counts = dataframe.select([
        F.sum(
            F.when(
                    F.col(c).isNull() | (F.trim(F.col(c)) == ""), 
                    1
            ).otherwise(0)
        ).alias(c)
        for c in dataframe.columns
    ]).collect()[0].asDict()

    # Lọc và hiển thị các cột có chứa null
    print(f"Tổng số dòng: {total_rows}\n")
    print(f"{'Tên cột':<25} | {'Số lượng Null':<15} | {'Tỷ lệ (%)':<10}")
    print("-" * 55)
    
    null_cols = {}
    for col_name, count in null_counts.items():
        if count > 0:
            percentage = round((count / total_rows) * 100, 2)
            print(f"{col_name:<25} | {count:<15} | {percentage}%")
            null_cols[col_name] = {"count": count, "percentage": percentage}
            
    return null_cols

# Sử dụng hàm:
null_report = check_null_summary(df)
print(null_report)

Tổng số dòng: 187531

Tên cột                   | Số lượng Null   | Tỷ lệ (%) 
-------------------------------------------------------
description               | 3023            | 1.61%
price_in_rupees           | 17664           | 9.42%
carpet_area               | 80670           | 43.02%
status                    | 615             | 0.33%
floor                     | 7079            | 3.77%
transaction               | 83              | 0.04%
furnishing                | 2899            | 1.55%
facing                    | 70233           | 37.45%
overlooking               | 81436           | 43.43%
society                   | 109677          | 58.48%
bathroom                  | 828             | 0.44%
balcony                   | 48936           | 26.09%
car_parking               | 103357          | 55.11%
ownership                 | 65517           | 34.94%
super_area                | 107685          | 57.42%
dimensions                | 187528          | 100.0%
plot_area               

In [5]:
df.select(['dimensions','plot_area']).show(5)

+----------+---------+
|dimensions|plot_area|
+----------+---------+
|      NULL|     NULL|
|      NULL|     NULL|
|      NULL|     NULL|
|      NULL|     NULL|
|      NULL|     NULL|
+----------+---------+
only showing top 5 rows


In [9]:
df.select(['bathroom','balcony','carpet_area']).show(5)

+--------+-------+-----------+
|bathroom|balcony|carpet_area|
+--------+-------+-----------+
|       1|      2|   500 sqft|
|       2|   NULL|   473 sqft|
|       2|   NULL|   779 sqft|
|       1|      1|   530 sqft|
|       2|   NULL|   635 sqft|
+--------+-------+-----------+
only showing top 5 rows


In [12]:
# 1. Ép kiểu trực tiếp cho price_in_rupees
df = df.withColumn("price_in_rupees", F.col("price_in_rupees").cast("double"))
# 2. Xử lý bathroom và balcony (thay "> 10" thành "10" rồi ép sang int)
for col_name in ["bathroom", "balcony"]:
    if col_name in df.columns:
        df = df.withColumn(
            col_name,
            F.when(
                # Bỏ mọi khoảng trắng thừa để bắt trọn cả ">10", "> 10", " >  10 "
                F.regexp_replace(F.trim(F.col(col_name)), r"\s+", "") == ">10", 
                "10"
            )
            .otherwise(F.col(col_name))
            .cast("double")  # Tránh lỗi biến "1.0", "2.0" thành null
            .cast("int")
        )
# 3. Xử lý carpet_area (tách bỏ chữ "sqft" lấy số)
if "carpet_area" in df.columns:
    df = df.withColumn(
        "carpet_area",
        F.regexp_extract(F.col("carpet_area"), r"(\d+(\.\d+)?)", 1).cast("double")
    )
# 4. Quy đổi amountin_rupees ra số tiền thực tế (Lac = * 1e5, Cr = * 1e7)
if "amountin_rupees" in df.columns:
    df = df.withColumn(
        "amount_clean",
        F.when(
            F.col("amountin_rupees").contains("Lac"),
            F.regexp_extract(F.col("amountin_rupees"), r"(\d+(\.\d+)?)", 1).cast("double") * 100000
        ).when(
            F.col("amountin_rupees").contains("Cr"),
            F.regexp_extract(F.col("amountin_rupees"), r"(\d+(\.\d+)?)", 1).cast("double") * 10000000
        ).otherwise(
            F.regexp_extract(F.col("amountin_rupees"), r"(\d+(\.\d+)?)", 1).cast("double")
        )
    )
# Kiểm tra lại Schema
df.printSchema()

root
 |-- index: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- description: string (nullable = true)
 |-- amountin_rupees: string (nullable = true)
 |-- price_in_rupees: double (nullable = true)
 |-- location: string (nullable = true)
 |-- carpet_area: double (nullable = true)
 |-- status: string (nullable = true)
 |-- floor: string (nullable = true)
 |-- transaction: string (nullable = true)
 |-- furnishing: string (nullable = true)
 |-- facing: string (nullable = true)
 |-- overlooking: string (nullable = true)
 |-- society: string (nullable = true)
 |-- bathroom: integer (nullable = true)
 |-- balcony: integer (nullable = true)
 |-- car_parking: string (nullable = true)
 |-- ownership: string (nullable = true)
 |-- super_area: string (nullable = true)
 |-- dimensions: string (nullable = true)
 |-- plot_area: string (nullable = true)
 |-- amount_clean: double (nullable = true)



In [ ]:
from pyspark.ml.feature import Imputer
# ==============================================================
# BƯỚC 1: XÓA CỘT CÓ TỶ LỆ NULL QUÁ CAO (> 60%) & CỘT KHÔNG CẦN THIẾT
# ==============================================================

# Tìm các cột có tỷ lệ null > 60% (như dimensions, plot_area...)
cols_to_drop = [
    col for col, info in null_report.items() 
    if info["percentage"] > 60
]

# Thêm các cột rác/cột chữ thô không dùng trực tiếp vào model
extra_drop = ["index", "title", "description", "amountin_rupees"]
cols_to_drop = list(set(cols_to_drop + [c for c in extra_drop if c in df.columns]))

print("Các cột sẽ bị xóa (> 60% null hoặc cột không dùng):", cols_to_drop)
df = df.drop(*cols_to_drop)


# ==============================================================
# BƯỚC 2: XÓA DÒNG BỊ THIẾU CỘT MỤC TIÊU (TARGET LABEL)
# ==============================================================
# Biến mục tiêu để dự đoán (ví dụ price_in_rupees) KHÔNG được điền bừa, thiếu thì phải bỏ dòng
target_col = "price_in_rupees"
df = df.filter(F.col(target_col).isNotNull() & (~F.isnan(target_col)))


# ==============================================================
# BƯỚC 3: DÙNG IMPUTER CỦA SPARK ML CHO CÁC CỘT SỐ (NUMERICAL)
# ==============================================================
# Các feature dạng số cần điền giá trị thiếu
# Khởi tạo Imputer với strategy="median" (hoặc "mean")
num_cols = [c for c in ["carpet_area", "bathroom", "balcony"] if c in df.columns]
imputer = Imputer(
    inputCols=num_cols,
    outputCols=num_cols,
    strategy="median"
)
df = imputer.fit(df).transform(df)

# ==============================================================
# BƯỚC 4: KIỂM TRA LẠI KẾT QUẢ
# ==============================================================
print(f"Số dòng còn lại sau khi lọc: {df.count()}")
df.printSchema()
df.select(num_cols + [target_col]).show(10)

Các cột sẽ bị xóa (> 60% null hoặc cột không dùng): ['plot_area', 'dimensions']


Số dòng còn lại sau khi lọc: 169866
root
 |-- price_in_rupees: double (nullable = true)
 |-- location: string (nullable = true)
 |-- carpet_area: double (nullable = true)
 |-- status: string (nullable = true)
 |-- floor: string (nullable = true)
 |-- transaction: string (nullable = true)
 |-- furnishing: string (nullable = true)
 |-- facing: string (nullable = true)
 |-- overlooking: string (nullable = true)
 |-- society: string (nullable = true)
 |-- bathroom: integer (nullable = true)
 |-- balcony: integer (nullable = true)
 |-- car_parking: string (nullable = true)
 |-- ownership: string (nullable = true)
 |-- super_area: string (nullable = true)

+-----------+--------+-------+---------------+
|carpet_area|bathroom|balcony|price_in_rupees|
+-----------+--------+-------+---------------+
|      500.0|       1|      2|         6000.0|
|      473.0|       2|      2|        13799.0|
|      779.0|       2|      2|        17500.0|
|      635.0|       2|      2|        18824.0|
|     1050.0

In [ ]:
print(null_report) # not processed
null_report = check_null_summary(df)
print(null_report)

{'description': {'count': 3023, 'percentage': 1.61}, 'price_in_rupees': {'count': 17664, 'percentage': 9.42}, 'carpet_area': {'count': 80670, 'percentage': 43.02}, 'status': {'count': 615, 'percentage': 0.33}, 'floor': {'count': 7079, 'percentage': 3.77}, 'transaction': {'count': 83, 'percentage': 0.04}, 'furnishing': {'count': 2899, 'percentage': 1.55}, 'facing': {'count': 70233, 'percentage': 37.45}, 'overlooking': {'count': 81436, 'percentage': 43.43}, 'society': {'count': 109677, 'percentage': 58.48}, 'bathroom': {'count': 828, 'percentage': 0.44}, 'balcony': {'count': 48936, 'percentage': 26.09}, 'car_parking': {'count': 103357, 'percentage': 55.11}, 'ownership': {'count': 65517, 'percentage': 34.94}, 'super_area': {'count': 107685, 'percentage': 57.42}, 'dimensions': {'count': 187528, 'percentage': 100.0}, 'plot_area': {'count': 187528, 'percentage': 100.0}}


Tổng số dòng: 169866

Tên cột                   | Số lượng Null   | Tỷ lệ (%) 
-------------------------------------------------------
status                    | 595             | 0.35%
floor                     | 6843            | 4.03%
transaction               | 6               | 0.0%
furnishing                | 2060            | 1.21%
facing                    | 62052           | 36.53%
overlooking               | 70646           | 41.59%
society                   | 97433           | 57.36%
car_parking               | 90950           | 53.54%
ownership                 | 58237           | 34.28%
super_area                | 93631           | 55.12%
{'status': {'count': 595, 'percentage': 0.35}, 'floor': {'count': 6843, 'percentage': 4.03}, 'transaction': {'count': 6, 'percentage': 0.0}, 'furnishing': {'count': 2060, 'percentage': 1.21}, 'facing': {'count': 62052, 'percentage': 36.53}, 'overlooking': {'count': 70646, 'percentage': 41.59}, 'society': {'count': 97433, 'percentage': 57.

In [27]:
df.show(20)

+---------------+--------+-----------+-------------+---------------+------------+--------------+------------+--------------------+--------------------+--------+-------+-----------+--------------------+----------+
|price_in_rupees|location|carpet_area|       status|          floor| transaction|    furnishing|      facing|         overlooking|             society|bathroom|balcony|car_parking|           ownership|super_area|
+---------------+--------+-----------+-------------+---------------+------------+--------------+------------+--------------------+--------------------+--------+-------+-----------+--------------------+----------+
|         6000.0|   thane|      500.0|Ready to Move|   10 out of 11|      Resale|   Unfurnished|        NULL|                NULL|Srushti Siddhi Ma...|       1|      2|       NULL|                NULL|      NULL|
|        13799.0|   thane|      473.0|Ready to Move|    3 out of 22|      Resale|Semi-Furnished|        East|         Garden/Park|         Dosti Vih

In [35]:
from pyspark.sql import functions as F
from pyspark.ml.feature import Imputer

# ==============================================================
# BƯỚC 1: DROP CÁC CỘT GÂY NHIỄU / TRÙNG LẶP (Nếu còn tồn tại)
# ==============================================================
cols_to_remove = [c for c in ["super_area", "society"] if c in df.columns]
if cols_to_remove:
    df = df.drop(*cols_to_remove)


# ==============================================================
# BƯỚC 2: FEATURE ENGINEERING CHO car_parking VÀ floor
# ==============================================================

# 2.1. car_parking: Chỉ xử lý nếu cột car_parking chưa bị drop
if "car_parking" in df.columns:
    df = df.withColumn(
        "parking_count",
        F.coalesce(
            F.expr("try_cast(regexp_extract(car_parking, '(\\\\d+)', 1) as int)"),
            F.lit(0)
        )
    ).drop("car_parking")

# 2.2. floor: Chỉ xử lý nếu cột floor chưa bị drop
if "floor" in df.columns:
    df = df.withColumn(
        "current_floor",
        F.when(F.lower(F.col("floor")).contains("ground"), 0)
         .otherwise(
             F.expr("try_cast(regexp_extract(floor, '(\\\\d+)', 1) as int)")
         )
    ).drop("floor")

    # Điền median cho các dòng thiếu current_floor (~4%)
    floor_imputer = Imputer(inputCols=["current_floor"], outputCols=["current_floor"], strategy="median")
    df = floor_imputer.fit(df).transform(df)


# ==============================================================
# BƯỚC 3: XỬ LÝ NULL CHO CÁC CỘT CHUỖI (CATEGORICAL)
# ==============================================================
cat_fill_values = {
    "status": "Ready to Move",
    "transaction": "Resale",
    "furnishing": "Unfurnished",
    "facing": "Unknown",
    "overlooking": "None",
    "ownership": "Unknown"
}

# Chỉ fillna cho các cột thực tế đang có trong df
df = df.fillna({k: v for k, v in cat_fill_values.items() if k in df.columns})


# ==============================================================
# BƯỚC 4: KIỂM TRA LẠI TỔNG THỂ DATAFRAME
# ==============================================================
print(f"Tổng số dòng: {df.count()}")
print("Các cột hiện có trong tập dữ liệu:")
df.printSchema()



Tổng số dòng: 169866
Các cột hiện có trong tập dữ liệu:
root
 |-- price_in_rupees: double (nullable = true)
 |-- location: string (nullable = true)
 |-- carpet_area: double (nullable = true)
 |-- status: string (nullable = false)
 |-- transaction: string (nullable = false)
 |-- furnishing: string (nullable = false)
 |-- facing: string (nullable = false)
 |-- overlooking: string (nullable = false)
 |-- bathroom: integer (nullable = true)
 |-- balcony: integer (nullable = true)
 |-- ownership: string (nullable = false)
 |-- parking_count: integer (nullable = false)
 |-- current_floor: integer (nullable = true)



In [36]:
df.show(20)


+---------------+--------+-----------+-------------+------------+--------------+------------+--------------------+--------+-------+--------------------+-------------+-------------+
|price_in_rupees|location|carpet_area|       status| transaction|    furnishing|      facing|         overlooking|bathroom|balcony|           ownership|parking_count|current_floor|
+---------------+--------+-----------+-------------+------------+--------------+------------+--------------------+--------+-------+--------------------+-------------+-------------+
|         6000.0|   thane|      500.0|Ready to Move|      Resale|   Unfurnished|     Unknown|                None|       1|      2|             Unknown|            0|           10|
|        13799.0|   thane|      473.0|Ready to Move|      Resale|Semi-Furnished|        East|         Garden/Park|       2|      2|            Freehold|            1|            3|
|        17500.0|   thane|      779.0|Ready to Move|      Resale|   Unfurnished|        East|  

In [37]:
from pyspark.ml.feature import StringIndexer

# Danh sách các cột chuỗi cần mã hóa thành số
categorical_cols = [
    "location", 
    "status", 
    "transaction", 
    "furnishing", 
    "facing", 
    "overlooking", 
    "ownership"
]

# Tên các cột đầu ra sau khi index (ví dụ: location -> location_idx)
indexed_cols = [f"{c}_idx" for c in categorical_cols]

# Khởi tạo StringIndexer
indexer = StringIndexer(
    inputCols=categorical_cols,
    outputCols=indexed_cols,
    handleInvalid="keep",        # 'keep': tự gán index riêng nếu gặp giá trị lạ/null, không làm crash app
    stringOrderType="frequencyDesc" # Giá trị xuất hiện nhiều nhất sẽ nhận index là 0.0
)

# Fit và transform
indexer_model = indexer.fit(df)
df_indexed = indexer_model.transform(df)

# Xem thử kết quả trước và sau khi index
df_indexed.select(
    "location", "location_idx", 
    "furnishing", "furnishing_idx", 
    "status", "status_idx"
).show(5)

+--------+------------+--------------+--------------+-------------+----------+
|location|location_idx|    furnishing|furnishing_idx|       status|status_idx|
+--------+------------+--------------+--------------+-------------+----------+
|   thane|        13.0|   Unfurnished|           1.0|Ready to Move|       0.0|
|   thane|        13.0|Semi-Furnished|           0.0|Ready to Move|       0.0|
|   thane|        13.0|   Unfurnished|           1.0|Ready to Move|       0.0|
|   thane|        13.0|   Unfurnished|           1.0|Ready to Move|       0.0|
|   thane|        13.0|   Unfurnished|           1.0|Ready to Move|       0.0|
+--------+------------+--------------+--------------+-------------+----------+
only showing top 5 rows


In [40]:
all_mappings = {
    col_name: {label: float(idx) for idx, label in enumerate(indexer_model.labelsArray[i])}
    for i, col_name in enumerate(categorical_cols)
}

# Ví dụ tra cứu thử cột 'status':
print("Mapping cột status:", all_mappings["status"])
print(all_mappings)

Mapping cột status: {'Ready to Move': 0.0}
{'location': {'new-delhi': 0.0, 'bangalore': 1.0, 'kolkata': 2.0, 'gurgaon': 3.0, 'ahmedabad': 4.0, 'hyderabad': 5.0, 'chennai': 6.0, 'jaipur': 7.0, 'greater-noida': 8.0, 'faridabad': 9.0, 'vadodara': 10.0, 'surat': 11.0, 'pune': 12.0, 'thane': 13.0, 'mumbai': 14.0, 'visakhapatnam': 15.0, 'zirakpur': 16.0, 'mohali': 17.0, 'chandigarh': 18.0, 'noida': 19.0, 'ranchi': 20.0, 'bhiwadi': 21.0, 'kochi': 22.0, 'goa': 23.0, 'dehradun': 24.0, 'ghaziabad': 25.0, 'lucknow': 26.0, 'patna': 27.0, 'sonipat': 28.0, 'guwahati': 29.0, 'kanpur': 30.0, 'navi-mumbai': 31.0, 'nagpur': 32.0, 'jamshedpur': 33.0, 'coimbatore': 34.0, 'vijayawada': 35.0, 'kalyan': 36.0, 'raipur': 37.0, 'palghar': 38.0, 'aurangabad': 39.0, 'agra': 40.0, 'bhubaneswar': 41.0, 'mangalore': 42.0, 'siliguri': 43.0, 'badlapur': 44.0, 'panchkula': 45.0, 'varanasi': 46.0, 'guntur': 47.0, 'nashik': 48.0, 'trivandrum': 49.0, 'thrissur': 50.0, 'ernakulam': 51.0, 'mysore': 52.0, 'allahabad': 53.0, 

In [41]:
# In bảng mapping cho TẤT CẢ các cột đã qua StringIndexer
print("=" * 60)
print(f"{'CỘT':<15} | {'GIÁ TRỊ GỐC (Chữ)':<25} | {'MÃ INDEX (Số)':<10}")
print("=" * 60)

# indexer_model.labelsArray chứa danh sách nhãn của từng cột theo đúng thứ tự categorical_cols
for col_idx, col_name in enumerate(categorical_cols):
    labels = indexer_model.labelsArray[col_idx]
    
    for num_idx, label in enumerate(labels):
        print(f"{col_name:<15} | {label:<25} | {float(num_idx):<10}")
    print("-" * 60)

CỘT             | GIÁ TRỊ GỐC (Chữ)         | MÃ INDEX (Số)
location        | new-delhi                 | 0.0       
location        | bangalore                 | 1.0       
location        | kolkata                   | 2.0       
location        | gurgaon                   | 3.0       
location        | ahmedabad                 | 4.0       
location        | hyderabad                 | 5.0       
location        | chennai                   | 6.0       
location        | jaipur                    | 7.0       
location        | greater-noida             | 8.0       
location        | faridabad                 | 9.0       
location        | vadodara                  | 10.0      
location        | surat                     | 11.0      
location        | pune                      | 12.0      
location        | thane                     | 13.0      
location        | mumbai                    | 14.0      
location        | visakhapatnam             | 15.0      
location        | zirakpur  

In [42]:
from pyspark.ml.feature import OneHotEncoder

# Mã hóa One-Hot các cột vừa index
ohe_cols = [f"{c}_vec" for c in categorical_cols]
encoder = OneHotEncoder(inputCols=indexed_cols, outputCols=ohe_cols)
df_encoded = encoder.fit(df_indexed).transform(df_indexed)

In [ ]:
df_encoded.show()

26/10/02 09:57:31 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+---------------+--------+-----------+-------------+------------+--------------+------------+--------------------+--------+-------+--------------------+-------------+-------------+------------+----------+---------------+--------------+----------+---------------+-------------+---------------+-------------+---------------+--------------+-------------+---------------+-------------+
|price_in_rupees|location|carpet_area|       status| transaction|    furnishing|      facing|         overlooking|bathroom|balcony|           ownership|parking_count|current_floor|location_idx|status_idx|transaction_idx|furnishing_idx|facing_idx|overlooking_idx|ownership_idx|   location_vec|   status_vec|transaction_vec|furnishing_vec|   facing_vec|overlooking_vec|ownership_vec|
+---------------+--------+-----------+-------------+------------+--------------+------------+--------------------+--------+-------+--------------------+-------------+-------------+------------+----------+---------------+--------------+-

In [ ]:
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# ==============================================================
# BƯỚC 1: LỌC OUTLIERS & TÍNH TỔNG TIỀN NHÀ (TOTAL_PRICE)
# ==============================================================
# Lọc bỏ giá trị bất thường (99% nhà ở Ấn Độ nằm trong khoảng này)
df_filtered = df_encoded.filter(
    (F.col("price_in_rupees") >= 1500) & (F.col("price_in_rupees") <= 35000) &
    (F.col("carpet_area") >= 150) & (F.col("carpet_area") <= 4000)
)

# Tính Tổng tiền nhà (Total Price = Đơn giá * Diện tích)
df_model_data = df_filtered.withColumn(
    "total_price", 
    F.col("price_in_rupees") * F.col("carpet_area")
)

print(f"Số lượng dòng dữ liệu hợp lệ: {df_model_data.count()}")
df_model_data.select("carpet_area", "price_in_rupees", "total_price").show(5)


# ==============================================================
# BƯỚC 2: GOM VECTOR ĐẶC TRƯNG (VECTOR ASSEMBLER)
# ==============================================================
# Các feature dùng để dự đoán 
num_features = ["carpet_area", "bathroom", "balcony", "parking_count", "current_floor"]
cat_features = [f"{c}_vec" for c in categorical_cols]
feature_cols = num_features + cat_features

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

final_dataset = assembler.transform(df_model_data).select("features", "total_price")


# ==============================================================
# BƯỚC 3: CHIA TẬP DỮ LIỆU & HUẤN LUYỆN LINEAR REGRESSION
# ==============================================================
train_data, test_data = final_dataset.randomSplit([0.8, 0.2], seed=42)

lr = LinearRegression(
    featuresCol="features", 
    labelCol="total_price",
    maxIter=100,
    regParam=0.1
)

lr_model = lr.fit(train_data)


# ==============================================================
# BƯỚC 4: DỰ ĐOÁN & ĐÁNH GIÁ KẾT QUẢ MỚI
# ==============================================================
predictions = lr_model.transform(test_data)

# Xem so sánh Giá thực tế vs Giá dự đoán
predictions.select(
    F.col("total_price").cast("long").alias("Thực tế (Rupees)"),
    F.col("prediction").cast("long").alias("Dự đoán (Rupees)"),
    F.abs(F.col("total_price") - F.col("prediction")).cast("long").alias("Chênh lệch")
).show(10)

# Đánh giá các chỉ số
evaluator_r2 = RegressionEvaluator(labelCol="total_price", predictionCol="prediction", metricName="r2")
evaluator_rmse = RegressionEvaluator(labelCol="total_price", predictionCol="prediction", metricName="rmse")
evaluator_mae = RegressionEvaluator(labelCol="total_price", predictionCol="prediction", metricName="mae")


r2 = evaluator_r2.evaluate(predictions)
rmse = evaluator_rmse.evaluate(predictions)
mae = evaluator_mae.evaluate(predictions)

print("=" * 45)
print(f"R² Score mới:                 {r2:.4f}  ({r2*100:.2f}%)")
print(f"RMSE (Sai số bình phương TB): {rmse:,.2f}")
print(f"MAE (Sai số tuyệt đối TB):     {mae:,.2f}")
print("=" * 45)

Số lượng dòng dữ liệu hợp lệ: 163962
+-----------+---------------+-----------+
|carpet_area|price_in_rupees|total_price|
+-----------+---------------+-----------+
|      500.0|         6000.0|  3000000.0|
|      473.0|        13799.0|  6526927.0|
|      779.0|        17500.0|  1.36325E7|
|      635.0|        18824.0| 1.195324E7|
|     1050.0|         6618.0|  6948900.0|
+-----------+---------------+-----------+
only showing top 5 rows


+----------------+----------------+----------+
|Thực tế (Rupees)|Dự đoán (Rupees)|Chênh lệch|
+----------------+----------------+----------+
|         9104550|        10268786|   1164236|
|         6705300|        11949743|   5244443|
|         6016500|        13505552|   7489052|
|        16528200|        16408363|    119836|
|        16528200|        16408363|    119836|
|        16528200|        16408363|    119836|
|        16528200|        16408363|    119836|
|        16528200|        16408363|    119836|
|        16528200|        16408363|    119836|
|        16528200|        16408363|    119836|
+----------------+----------------+----------+
only showing top 10 rows


R² Score mới:                 0.7201  (72.01%)
RMSE (Sai số bình phương TB): 4,198,014.09
MAE (Sai số tuyệt đối TB):     2,768,039.62


In [56]:
print(f"Hệ số chặn (Intercept): {lr_model.intercept}")
print(f"Số lượng hệ số góc (Coefficients): {len(lr_model.coefficients)}")

Hệ số chặn (Intercept): -7815460.930645398
Số lượng hệ số góc (Coefficients): 127


## Bản nâng cấp số 1: Biến đổi Logarit nhãn mục tiêu ($\ln(y)$)

### 1. Cơ sở lý thuyết:
- **Phân phối Log-normal:** Trong kinh tế và bất động sản, giá nhà luôn bị lệch phải (*Right-skewed*) với đuôi dài hướng về phía các căn biệt thự/penthouse siêu đắt. Việc áp dụng biến đổi logarit tự nhiên:
  $$ y_{\text{log}} = \ln(\text{total\_price}) $$
  giúp nén các giá trị cực lớn, đưa phân phối của biến mục tiêu về dạng chuẩn đối xứng (*Gaussian-like distribution*).
- **Ổn định phương sai sai số (Homoscedasticity):** Một giả định then chốt của Hồi quy tuyến tính (*Gauss-Markov Theorem*) là phương sai sai số không đổi ($\operatorname{Var}(\varepsilon | X) = \sigma^2$). Biến đổi logarit giúp triệt tiêu hiện tượng phương sai thay đổi (*Heteroskedasticity*).
- **Khôi phục giá trị thực tế (Inverse Transform):** Khi dự đoán trên thang log $\hat{y}_{\text{log}}$, ta lấy hàm mũ để khôi phục về tiền thực tế:
  $$ \hat{y}_{\text{actual}} = \exp(\hat{y}_{\text{log}}) $$

In [61]:
from pyspark.sql import functions as F
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# ==============================================================
# BƯỚC 1: BIẾN ĐỔI LOGARIT NHÃN MỤC TIÊU: ln(total_price)
# ==============================================================
df_log_data = df_model_data.withColumn(
    "log_total_price", 
    F.log(F.col("total_price"))
)

# Chuẩn bị dataset với nhãn log
final_log_dataset = assembler.transform(df_log_data).select(
    "features", "log_total_price", "total_price"
)

# ==============================================================
# BƯỚC 2: CHIA TẬP DỮ LIỆU TRAIN / TEST (80% - 20%)
# ==============================================================
train_log, test_log = final_log_dataset.randomSplit([0.8, 0.2], seed=42)

# ==============================================================
# BƯỚC 3: HUẤN LUYỆN LINEAR REGRESSION TRÊN THANG ĐO LOGARIT
# ==============================================================
lr_log = LinearRegression(
    featuresCol="features", 
    labelCol="log_total_price",
    maxIter=100,
    regParam=0.1
)

lr_log_model = lr_log.fit(train_log)

# ==============================================================
# BƯỚC 4: DỰ ĐOÁN & CHUYỂN ĐỔI NGƯỢC (exp(prediction))
# ==============================================================
predictions_log = lr_log_model.transform(test_log)

# Khôi phục về giá tiền thực tế bằng hàm mũ: exp(log_pred)
predictions_final = predictions_log.withColumn(
    "pred_actual_price", 
    F.exp(F.col("prediction"))
)

# Hiển thị 10 kết quả so sánh
predictions_final.select(
    F.col("total_price").cast("long").alias("Thực tế (INR)"),
    F.col("pred_actual_price").cast("long").alias("Dự đoán Exp (INR)"),
    F.abs(F.col("total_price") - F.col("pred_actual_price")).cast("long").alias("Chênh lệch")
).show(10)

# ==============================================================
# BƯỚC 5: ĐÁNH GIÁ CHỈ SỐ R2 VÀ SAI SỐ
# ==============================================================
# 1. Trên thang đo Logarit
eval_r2_log = RegressionEvaluator(labelCol="log_total_price", predictionCol="prediction", metricName="r2")
eval_rmse_log = RegressionEvaluator(labelCol="log_total_price", predictionCol="prediction", metricName="rmse")

r2_log = eval_r2_log.evaluate(predictions_log)
rmse_log = eval_rmse_log.evaluate(predictions_log)

# 2. Trên thang đo tiền thực tế (Rupees)
eval_r2_real = RegressionEvaluator(labelCol="total_price", predictionCol="pred_actual_price", metricName="r2")
eval_rmse_real = RegressionEvaluator(labelCol="total_price", predictionCol="pred_actual_price", metricName="rmse")
eval_mae_real = RegressionEvaluator(labelCol="total_price", predictionCol="pred_actual_price", metricName="mae")

r2_real = eval_r2_real.evaluate(predictions_final)
rmse_real = eval_rmse_real.evaluate(predictions_final)
mae_real = eval_mae_real.evaluate(predictions_final)

print("=" * 60)
print(f"1. Trên thang đo Log (ln(y)): R² = {r2_log:.4f} ({r2_log*100:.2f}%) | RMSE = {rmse_log:.4f}")
print(f"2. Trên thang đo tiền thực tế (Rupees sau khi exp):")
print(f"   - R² Score: {r2_real:.4f} ({r2_real*100:.2f}%)")
print(f"   - RMSE:     {rmse_real:,.2f} INR")
print(f"   - MAE:      {mae_real:,.2f} INR")
print("=" * 60)


+-------------+-----------------+----------+
|Thực tế (INR)|Dự đoán Exp (INR)|Chênh lệch|
+-------------+-----------------+----------+
|      9104550|          6757737|   2346812|
|      6705300|          8433764|   1728464|
|      6016500|          8916485|   2899985|
|     16528200|         12452201|   4075998|
|     16528200|         12452201|   4075998|
|     16528200|         12452201|   4075998|
|     16528200|         12452201|   4075998|
|     16528200|         12452201|   4075998|
|     16528200|         12452201|   4075998|
|     16528200|         12452201|   4075998|
+-------------+-----------------+----------+
only showing top 10 rows


1. Trên thang đo Log (ln(y)): R² = 0.7270 (72.70%) | RMSE = 0.3721
2. Trên thang đo tiền thực tế (Rupees sau khi exp):
   - R² Score: 0.6759 (67.59%)
   - RMSE:     4,517,548.36 INR
   - MAE:      2,479,468.06 INR
